# SFINCS Scenario Stats

> **Stage Contract**
>
> Requires: scenario folders and completed run_outputs/evt_* results
>
> Produces: scenario statistics and evaluation products under data/sfincs/stats
>
> Next: publish results or iterate on source/scenario settings

In [ ]:
# Load local packages and this Location Workspace.
import sys
from pathlib import Path

repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))
workspace = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
config_yaml = workspace / "config.yaml"

# Load notebook tools.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from IPython.display import display

from sfincs_runs.config import load_runtime
from sfincs_runs.scenarios import scenario_stats as stats

config, paths = load_runtime(config_yaml)
pd.set_option("display.max_columns", 80)


## Step 0: Load Stats Inputs

Stats are not raw SFINCS outputs alone. They need three folder families:

- `scenarios_root`: prepared SFINCS event folders with `sfincs.inp`, `sfincs.bzs`, and manifests.
- `storage_root`: persisted solver outputs, especially `sfincs_map.nc`.
- `design_outputs_root`: `design_events` outputs that explain the sampled peak, template, return period, and SLR scenario.

These paths come from `sfincs_runs/project.yaml` through `build_paths()`.

In [ ]:
settings = pd.Series(
    {
        "scenarios_root": str(paths["scenarios_root"]),
        "storage_root": str(paths["storage_root"]),
        "stats_root": str(paths["stats_root"]),
        "design_outputs_root": str(paths["design_outputs_root"]),
        "land_threshold_m": 0.0,
        "huthresh_m": 0.01,
        "impact_threshold_m": 0.10,
    },
    name="value",
)
settings

## Step 1

- Which SLR scenario generated this run?
- What return period was sampled?
- Which historical hydrograph template shaped the event?
- Does the SFINCS boundary file match the expected design-event peak?

In [ ]:
scenario_summary, scenario_rows = stats.load_scenario_build(paths["scenarios_root"])
design_rows, design_attrs = stats.load_design_events(scenario_summary)

display(pd.Series(scenario_summary, name="scenario_build"))
display(pd.Series(design_attrs, name="design_event_attrs"))

design_frame = pd.DataFrame.from_dict(design_rows, orient="index")
cols = [c for c in ["sample_rp_years", "peak_m", "absolute_peak_m", "template_id", "template_peak_time", "tail_morph_factor"] if c in design_frame]
design_frame[cols].head()

## Step 2: Pick Completed Events

A scenario folder is prepared input. A completed run also has `sfincs_map.nc` in the storage folder.

The next cell finds prepared events and keeps only events with map output, so the notebook can run safely on a subset.

In [ ]:
all_events = stats.event_dirs(paths["scenarios_root"])
completed_events = [d for d in all_events if (paths["storage_root"] / d.name / "sfincs_map.nc").exists()]

LIMIT = 12  # set to None for all completed events
selected_events = completed_events[:LIMIT] if LIMIT is not None else completed_events

pd.Series(
    {
        "prepared_event_count": len(all_events),
        "completed_event_count": len(completed_events),
        "selected_event_count": len(selected_events),
        "first_selected_event": selected_events[0].name if selected_events else None,
    }
)

## Step 3: Inspect One Event End to End

For one event, compare the design-event forcing metadata with SFINCS results.

Important metric logic:
- `zs - zb` gives water depth.
- land cells are active cells with bed elevation above `land_threshold_m`.
- baseline wet land at `t0` is not counted as new impact.
- incremental flood depth measures added land depth above the `t0` baseline.
- impact extent uses the configured `impact_threshold_m`.

In [ ]:
if not selected_events:
    raise RuntimeError("No completed SFINCS events found. Run scenarios first.")

event_dir = selected_events[0]
row = stats.event_stats(
    event_dir,
    paths["storage_root"],
    settings["land_threshold_m"],
    settings["huthresh_m"],
    settings["impact_threshold_m"],
    scenario_summary,
    scenario_rows,
    design_rows,
    design_attrs,
)

focus = [
    "event_id", "design_scenario", "design_slr_offset_m", "sample_rp_years", "template_id",
    "driver_h_magnitude", "expected_bzs_peak_max_m", "bzs_peak_max_m",
    "peak_incremental_land_depth_m", "peak_incremental_flooded_area_km2",
    "longest_incremental_flood_duration_h", "area_incremental_flooded_ge_24h_km2",
]
pd.Series({k: row.get(k) for k in focus}, name=event_dir.name)

## Step 4: Visualize the One-Event Flood Signal

- left: max incremental land depth per cell.
- right: wet land cell count through time.

In [ ]:
inp = stats.parse_sfincs_inp(event_dir / "sfincs.inp")
with xr.open_dataset(row["map_path"], decode_times=False) as ds:
    zs = np.asarray(ds["zs"].values, np.float32)
    zb = np.asarray(ds["zb"].values, np.float32)
    active = np.asarray(ds["msk"].values, float) > 0
    hours, dt = stats.parse_time(ds, inp)

depth = np.where(active[None, :, :], zs - zb[None, :, :], np.nan)
land = active & np.isfinite(zb) & (zb > settings["land_threshold_m"])
baseline = np.where(land, np.maximum(depth[0], 0.0), np.nan)
incremental = np.where(land[None, :, :], np.maximum(depth - np.nan_to_num(baseline, nan=0.0)[None, :, :], 0.0), np.nan)
impact = np.isfinite(incremental) & (incremental > settings["impact_threshold_m"])

peak_map = np.full(land.shape, np.nan, float)
impacted_cells = np.any(impact, axis=0)
if np.any(impacted_cells):
    peak_map[impacted_cells] = np.nanmax(np.where(impact, incremental, np.nan)[:, impacted_cells], axis=0)
impact_counts = impact.sum(axis=(1, 2))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
im = axes[0].imshow(peak_map, origin="lower", cmap="viridis")
axes[0].set_title(f"{event_dir.name}: peak incremental land depth [m]")
fig.colorbar(im, ax=axes[0], shrink=0.75)

axes[1].plot(hours, impact_counts, lw=1.5)
axes[1].set_title("Impacted land cells through time")
axes[1].set_xlabel("hour since start")
axes[1].set_ylabel("cell count")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Step 5: Build the Stats Table

In [ ]:
rows = []
for d in selected_events:
    rows.append(
        stats.event_stats(
            d,
            paths["storage_root"],
            settings["land_threshold_m"],
            settings["huthresh_m"],
            settings["impact_threshold_m"],
            scenario_summary,
            scenario_rows,
            design_rows,
            design_attrs,
        )
    )

df = pd.DataFrame(rows).sort_values("event_id").reset_index(drop=True)
display(df[[
    "event_id", "design_scenario", "design_slr_offset_m", "sample_rp_years",
    "expected_bzs_peak_max_m", "bzs_peak_max_m",
    "peak_incremental_land_depth_m", "peak_incremental_flooded_area_km2",
    "longest_incremental_flood_duration_h",
]].head())

df.describe(include="all").T.head(20)

## Step 6: Check Alignment Between Inputs and Flood Outputs

- boundary peak from `sfincs.bzs` should track the design-event expected boundary peak.
- return period and SLR scenario should be available beside flood metrics.
- flood extent/depth rankings should point back to event IDs for map inspection.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(df["expected_bzs_peak_max_m"], df["bzs_peak_max_m"], s=24, alpha=0.8)
axes[0].set_title("Expected vs written boundary peak")
axes[0].set_xlabel("expected bzs peak [m]")
axes[0].set_ylabel("written bzs peak [m]")
axes[0].grid(True, alpha=0.3)

axes[1].scatter(df["sample_rp_years"], df["peak_incremental_land_depth_m"], s=24, alpha=0.8)
axes[1].set_xscale("log")
axes[1].set_title("Return period vs flood depth")
axes[1].set_xlabel("sample return period [years]")
axes[1].set_ylabel("peak incremental land depth [m]")
axes[1].grid(True, alpha=0.3)

axes[2].scatter(df["peak_incremental_land_depth_m"], df["peak_incremental_flooded_area_km2"], s=24, alpha=0.8)
axes[2].set_title("Depth vs extent")
axes[2].set_xlabel("peak incremental land depth [m]")
axes[2].set_ylabel("peak incremental flood extent [km^2]")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

df.sort_values("peak_incremental_land_depth_m", ascending=False)[["event_id", "design_scenario", "sample_rp_years", "peak_incremental_land_depth_m", "peak_incremental_flooded_area_km2"]].head(10)

## Step 7: Write Notebook Outputs

In [ ]:
outdir = paths["stats_root"] / "notebook"
outdir.mkdir(parents=True, exist_ok=True)

csv_path = outdir / "scenario_stats_notebook.csv"
df.to_csv(csv_path, index=False)

metric_cols = [
    "zsini_m", "baseline_t0_flooded_area_km2", "peak_incremental_land_depth_m",
    "peak_incremental_flooded_area_km2", "anytime_incremental_flooded_area_km2",
    "longest_incremental_flood_duration_h", "mean_incremental_flood_duration_h",
    "area_incremental_flooded_ge_24h_km2",
]
summary = {
    "event_count": int(len(df)),
    "design_outputs_root": scenario_summary.get("design_outputs_root"),
    "design_scenarios": sorted(str(x) for x in df["design_scenario"].dropna().unique()),
    "design_slr_offsets_m": sorted(float(x) for x in pd.to_numeric(df["design_slr_offset_m"], errors="coerce").dropna().unique()),
    "metric_summaries": {c: stats.series_summary(df, c) for c in metric_cols},
}

(outdir / "summary_notebook.json").write_text(__import__("json").dumps(summary, indent=2), encoding="utf-8")
pd.Series({"csv": str(csv_path), "summary": str(outdir / "summary_notebook.json")})